# 02 — Baseline Pruner Evaluation Results

Run all baseline pruning strategies and compare results.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.evaluation.runner import EvaluationRunner
from src.utils.io import load_jsonl
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load data
samples = load_jsonl("../data/processed/hotpotqa_experiments.jsonl")[:20]
print(f"Loaded {len(samples)} samples for evaluation")

In [ ]:
# Run all baseline pruners with dry_run mode
pruners = ["no_pruning", "naive_truncation", "recomp"]
results_by_pruner = {}

for pruner in pruners:
    print(f"\nEvaluating {pruner}...")
    output_path = f"../experiments/results/{pruner}_results.jsonl"
    runner = EvaluationRunner(
        pruner_name=pruner,
        generator_model="dry_run",
        judge_model="dry_run",
        output_path=output_path,
    )
    agg = runner.run(samples)
    results_by_pruner[pruner] = agg
    print(f"  Faithfulness: {agg['faithfulness_mean']:.3f}")
    print(f"  Compression: {agg['compression_ratio_mean']:.3f}")

In [ ]:
# Create comparison table
df = pd.DataFrame([
    {
        'Pruner': name,
        'Faithfulness': agg['faithfulness_mean'],
        'EM': agg['em_mean'],
        'F1': agg['f1_mean'],
        'Compression': agg['compression_ratio_mean'],
        'Latency (s)': agg['latency_mean_s'],
    }
    for name, agg in results_by_pruner.items()
])

print("\n" + "="*80)
print(df.to_string(index=False))
print("="*80)

In [ ]:
# Plot: Faithfulness vs Compression
plt.figure(figsize=(8, 6))
plt.scatter(
    df['Compression'],
    df['Faithfulness'],
    s=200,
    alpha=0.7,
    c=range(len(df)),
    cmap='viridis'
)
for i, row in df.iterrows():
    plt.annotate(row['Pruner'], (row['Compression'], row['Faithfulness']),
                xytext=(5, 5), textcoords='offset points')

plt.xlabel('Compression Ratio')
plt.ylabel('Faithfulness Score')
plt.title('Pruning Strategies: Faithfulness vs Compression')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../experiments/results/faithfulness_vs_compression.png', dpi=150)
plt.show()